# Calibrating the advice constants

The advice view (Spend / Save tags in Let It Ride) values a golfer's **future** windows with two judgment calls in `scripts/predictions/advice_values.py`:

| Constant | Current | Meaning |
|---|---|---|
| `P_PLAYS_OPEN` | 0.60 | chance he enters a later *open* event |
| `P_PLAYS_RESTRICTED` | 0.90 | chance he enters a later major / signature / playoff he qualifies for |
| `DECAY_PER_WEEK` | 0.98 | how fast "what he's worth now" stops predicting "what he'll be worth then" |

This notebook measures them from the 2026 season instead of guessing.

**Data:** `data/analysis/advice_calibration_pairs.csv` — one row per (golfer, a week we predicted, a later event). Rebuild it any time with `python3 scripts/analysis/build_advice_calibration.py`.


In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/analysis/advice_calibration_pairs.csv")
print(f"{len(df):,} pairs")
df.head()
df['open_event'] = ~df['restricted_b']
df['played'] = df['played_b']

23,445 pairs


## 1. How often does a golfer actually play a later event?

`played_b` is True when he teed it up at event B.

**Task:** compute the play rate split by `restricted_b` (open vs restricted events).

Then split restricted events by whether he's likely *qualified*. We don't have the qualification list here, so use world rank as the proxy the advice uses: `world_rank <= 60`.

Hint: `df.groupby(...)["played_b"].mean()` — and look at the counts too (`.agg(["mean", "count"])`) so you know how much to trust each number.

In [14]:
# TODO: play rate for open vs restricted events
print(f"Open events: {len(df[df['open_event'] == True]):,}")
played_by_open = df[df['open_event'] == True]['played'].sum()
print(f"Played by open events: {played_by_open:,} ({played_by_open / len(df[df['open_event'] == True]):.1%})")
print(f"Restricted events: {len(df[df['open_event'] == False]):,}")
played_by_restricted = df[df['open_event'] == False]['played'].sum()
print(f"Played by restricted events: {played_by_restricted:,} ({played_by_restricted / len(df[df['open_event'] == False]):.1%})")
df.groupby('open_event')['played'].mean().reset_index().rename(columns={'played': 'play_rate'})
df.groupby('open_event')['played'].mean().reset_index().rename(columns={'played': 'play_rate'}).to_csv("../../data/analysis/advice_calibration_play_rates.csv", index=False)


Open events: 11,048
Played by open events: 5,529 (50.0%)
Restricted events: 12,397
Played by restricted events: 4,678 (37.7%)


In [15]:
# TODO: restricted events only — play rate for world_rank <= 60 vs > 60
restricted_df = df[~df["open_event"]]
restricted_df.groupby(restricted_df["world_rank"] <= 60)["played"].agg(["mean", "count"])
print(f"Restricted events: {len(restricted_df):,}")
played_by_top_60 = restricted_df[restricted_df['world_rank'] <= 60]['played'].agg(['mean', 'count', 'sum'])
played_by_below_60 = restricted_df[restricted_df['world_rank'] > 60]['played'].agg(['mean', 'count', 'sum'])
restricted_df.groupby('world_rank')['played'].mean().reset_index().rename(columns={'played': 'play_rate'})
restricted_df.groupby('world_rank')['played'].mean().reset_index().rename(columns={'played': 'play_rate'}).to_csv("../../data/analysis/advice_calibration_restricted_play_rates.csv", index=False)

Restricted events: 12,397


**Think about it:** does the play rate depend on how good the golfer is, even at open events? A world #5 skips most opposite-field events; a #120 plays nearly everything he can get into. Try bucketing `world_rank` (e.g. `pd.cut(df.world_rank, [0, 10, 30, 60, 120, 1000])`) and see whether one constant is enough or the advice should use a few.

In [16]:
bins = [0, 10, 30, 60, 120, 1000]
df["rank_bucket"] = pd.cut(df["world_rank"], bins)
df.groupby(["rank_bucket", "open_event"], observed=True)["played"].agg(["mean", "count"]).unstack()

mean           count      
open_event      False     True  False True 
rank_bucket                                
(0, 10]      0.955665  0.179795   609   584
(10, 30]     0.869868  0.259751  1291  1205
(30, 60]     0.745509  0.396681  2004  1868
(60, 120]    0.373266  0.670605  3172  2793
(120, 1000]  0.055441  0.543062  5321  4598

## 2. How fast does today's value go stale?

When a golfer *did* play B and B had a Tuesday prediction (`ev_b_model` not null), compare:

- `ev_extrap` — what the advice would have predicted for him at B from week A (purse + field-strength adjusted, **before** decay)
- `ev_b_model` — what the model actually said about him on B's Tuesday

If form were permanent, `ev_b_model / ev_extrap` would average ~1.0 at every distance. Decay means the ratio drifts as `weeks_apart` grows.

**Task:** filter to rows with both numbers, compute the ratio, and look at its **median** by `weeks_apart` (rounded to whole weeks). Median, not mean — a few big ratios from long shots will skew a mean.

In [17]:
ratio_df = df[df["ev_extrap"] > 0].copy()
ratio_df["ev_ratio"] = ratio_df["ev_b_model"] / ratio_df["ev_extrap"]
ratio_df = ratio_df.dropna(subset=["ev_ratio"])
ratio_df["weeks_apart_rounded"] = ratio_df["weeks_apart"].round().astype(int)
ratio_df.groupby("weeks_apart_rounded")["ev_ratio"].agg(["median", "count"])

,median,count
weeks_apart_rounded,,
1,1.015651,954
2,0.965666,769
3,1.035465,859
4,1.066408,794
5,0.958074,831
6,1.016203,749
7,1.000311,711
8,1.031962,748
9,0.963596,576


**Fit the decay:** if value decays by a constant factor `d` per week, then `median_ratio ≈ d ** weeks`, so `log(median_ratio) ≈ weeks × log(d)`. A straight-line fit through the origin of `log(ratio)` against `weeks` gives `log(d)`.

Hint: `slope = (w * y).sum() / (w * w).sum()` is the least-squares slope through the origin for x=`w`, y=`y`. Then `d = np.exp(slope)`.

In [18]:
# TODO: fit d, then compare with the current 0.98
med = ratio_df.groupby("weeks_apart_rounded")["ev_ratio"].median()
w = med.index.values.astype(float)
y = np.log(med.values)
d = np.exp((w * y).sum() / (w * w).sum())
print(f"decay per week: {d:.4f}")
from scipy.optimize import curve_fit
def linear_func(x, a, b):
    return a * x + b
popt, pcov = curve_fit(linear_func, med.index, y)
slope = popt[0]
intercept = popt[1]
print(f"Fitted line: y = {slope:.4f}x + {intercept:.4f}")

decay per week: 0.9992
Fitted line: y = -0.0002x + -0.0068


## 3. Sanity check against real money (optional)

`earnings_b` is what he actually earned at B. For golfers who played, compare the mean of `earnings_b` with the mean of `ev_extrap × your decay` by weeks_apart. Golf payouts are lumpy (one win dwarfs everything), so expect noise — this is a smell test, not the fit.

## 4. Write down what you found

Fill in the numbers you'd put in `advice_values.py` and why. Then we'll change the constants, rerun Bank of Utah's advice, and see how the Spend / Save split moves.

| Constant | Old | New | Evidence (n) |
|---|---|---|---|
| P_PLAYS_OPEN | 0.60 | | |
| P_PLAYS_RESTRICTED | 0.90 | | |
| DECAY_PER_WEEK | 0.98 | | |
